# Regressions following the supervisor's suggestions

This notebook **reads the clean dataset** `sea_clean_firms.csv` (one row per firm, built from the World Bank Enterprise Surveys) and runs every regression the supervisor asked for. **No data cleaning happens here**, so every result uses exactly the same firms.

| Part | Supervisor's suggestion | What is estimated |
|---|---|---|
| **1** | Main results, **both size definitions** | RQ1 (score), each practice, RQ2 to RQ4, for SMEs under the national ceilings and under the World Bank's 100+ definition |
| **2** | **Weighted** regressions | The same results unweighted, with weights rescaled within each survey, and rescaled + capped |
| **3** | **Indonesia + Malaysia vs the rest** | The payoff of management in each group, for all SMEs and for SMEs with 20+ employees (like-for-like) |
| **4** | **SMEs vs large firms** | The payoff in each group and the difference, under both definitions |
| **5** | One chart | The main payoff across every version, at a glance |
| **6** | Formal training (supplementary) | Do firms that train sell more, and does management pay off more with training? |
| **7** | Regression tables (stargazer) | The main models in journal format, saved as `.html` and `.tex` |

**Every model compares firms only within the same country and year** (country-year fixed effects), with robust standard errors. Key results also get a **wild-cluster bootstrap p-value** (Webb weights, clustered by the 12 surveys), the most honest test with few surveys.

**Reading:** a score coefficient β means firms with a score 0.1 higher sell about $(e^{0.1\beta}-1)\times100\%$ more than similar firms; a yes/no coefficient β means $(e^{\beta}-1)\times100\%$ more.

**What it can and cannot show:** differences between similar firms, not that management causes higher sales.

**How to run:** open the notebook in Google Colab and choose `Runtime → Run all`. When the Upload button appears, choose `sea_clean_firms.csv`. The bootstrap takes several minutes. The reports are saved in your Google Drive (`wbes_out/sea_data/q7/`), or in the Colab session if Drive is not connected.

### Step 0. Install the bootstrap package (run once)

In [ ]:
!pip install -q wildboottest stargazer

### Step 1. Load the clean dataset and set up
Reads `sea_clean_firms.csv` and keeps the **final sample** (management score and believable sales). Builds the yes/no practice variables from the raw answers.

In [ ]:
import re, io, base64, warnings
import numpy as np, pandas as pd
from pathlib import Path
import statsmodels.formula.api as smf
from wildboottest.wildboottest import wildboottest
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30, "display.width", 220)

# ---- Where are the results saved? In Google Drive if it is connected, otherwise in this Colab session
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    OUT = Path('/content/drive/MyDrive/wbes_out/sea_data/q7')                          # results of this notebook
except Exception:
    OUT = Path('/content/q7_results')                                                  # no Drive: results stay in this session
OUT.mkdir(parents=True, exist_ok=True)

# ---- Where is the clean dataset? Tried in this order:
#   1. the Drive folder where the Question 1 notebook saves it
#   2. a location typed in OTHER_PATH below (optional)
#   3. a copy uploaded to this Colab session (Files panel on the left)
#   4. otherwise an Upload button appears: choose sea_clean_firms.csv from your computer
CLEAN_FILE = Path('/content/drive/MyDrive/wbes_out/sea_data/q1/sea_clean_firms.csv')   # made by the Question 1 notebook
OTHER_PATH = ""        # optional, for example '/content/drive/MyDrive/thesis/sea_clean_firms.csv'
candidates = [CLEAN_FILE, Path(OTHER_PATH) if OTHER_PATH else None, Path('/content/sea_clean_firms.csv')]
found = next((p for p in candidates if p is not None and p.exists()), None)
if found is None:
    from google.colab import files
    print("sea_clean_firms.csv was not found. Please choose it from your computer:")
    uploaded = files.upload()                                                          # saves the file in /content
    found = Path('/content') / next(iter(uploaded))
CLEAN_FILE = found
print("Using the clean dataset:", CLEAN_FILE)
print("Results will be saved in:", OUT)
B_REPS = 4999                                                                           # bootstrap repetitions

ALL = pd.read_csv(CLEAN_FILE)
D = ALL[ALL["final_sample"]].copy()                                  # firms with a score and believable sales
D["score"] = D["score_B"]                                            # main management score
yes = lambda q: (D[q] == 1).astype(float).where(D[q + "_status"] == "answered")
D["monitoring"], D["targets"], D["bonuses"] = yes("r2"), yes("r4"), yes("r8")
D["female"], D["credit"] = D["female_owner"], D["credit_obstacle"]
D["log_age"] = np.log1p(D["firm_age"])
D["log_emp"] = D["log_employees"]

REPORT, SUMMARY = {}, {}
def add(title, df, note=""):
    """Show a table and keep it for the saved report."""
    REPORT[title] = (df.reset_index(drop=True), note)
    print("\n" + title + (f"\n  ({note})" if note else ""))
    display(df)

def add_figure(title, fig, note=""):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
    REPORT[title] = ("__figure__", note, base64.b64encode(buf.getvalue()).decode())
    print("\n" + title); plt.show(); plt.close(fig)

DEFS = {"National ceilings": "large_national", "World Bank 100+": "large_wb"}
SME = {name: D[D[col] == 0] for name, col in DEFS.items()}
print(f"Final sample: {len(D):,} firms. SMEs: " + ", ".join(f"{k} {len(v):,}" for k, v in SME.items()))

### Step 2. Tools used by every model

In [ ]:
FE = " + C(survey)"
CONTROLS = " + foreign + exporter + training + log_age"
WEIGHTS = {"Unweighted": None, "Weights rescaled within survey": "weight_rescaled",
           "Rescaled and capped weights": "weight_rescaled_capped"}

def run(formula, needed, data, wcol=None):
    """OLS (or weighted least squares) with robust standard errors, on firms with every needed variable."""
    d = data.dropna(subset=needed + ["survey"] + ([wcol] if wcol else [])).copy()
    m = (smf.wls(formula, data=d, weights=d[wcol]) if wcol else smf.ols(formula, data=d)).fit(cov_type="HC1")
    return m, d

def boot_p(formula, d, term):
    """Wild-cluster bootstrap p-value (Webb weights, clustered by survey)."""
    res = wildboottest(smf.ols(formula, data=d), B=B_REPS, cluster=pd.factorize(d["survey"])[0], param=term,
                       weights_type="webb", impose_null=True, bootstrap_type="11", seed=12345, parallel=False, show=False)
    return float(res["p-value"].iloc[0])

def combo(m, weights):
    """Estimate and p-value of a sum of coefficients."""
    v = np.zeros(len(m.params))
    for t, w in weights.items():
        v[list(m.params.index).index(t)] = w
    r = m.t_test(v.reshape(1, -1))
    return float(np.squeeze(r.effect)), float(np.squeeze(r.pvalue))

pct = lambda b, step: 100 * (np.exp(step * b) - 1)
def evidence(p):
    if p is None or pd.isna(p):
        return ""
    return "very strong" if p < 0.01 else "strong" if p < 0.05 else "weak" if p < 0.10 else "no clear evidence"

# The eight key results: (label, formula, needed variables, term, step for the % reading, kind)
KEY = [
    ("RQ1: management score", "log_sales ~ score + log_emp" + FE, ["log_sales", "score", "log_emp"], "score", 0.1, "per +0.1 score"),
    ("RQ1 with controls", "log_sales ~ score + log_emp" + CONTROLS + FE,
     ["log_sales", "score", "log_emp", "foreign", "exporter", "training", "log_age"], "score", 0.1, "per +0.1 score"),
    ("Monitoring (r2)", "log_sales ~ monitoring + targets + bonuses + log_emp" + FE,
     ["log_sales", "monitoring", "targets", "bonuses", "log_emp"], "monitoring", 1, "vs similar firms without"),
    ("Targets (r4)", "log_sales ~ monitoring + targets + bonuses + log_emp" + FE,
     ["log_sales", "monitoring", "targets", "bonuses", "log_emp"], "targets", 1, "vs similar firms without"),
    ("Bonuses for managers (r8)", "log_sales ~ monitoring + targets + bonuses + log_emp" + FE,
     ["log_sales", "monitoring", "targets", "bonuses", "log_emp"], "bonuses", 1, "vs similar firms without"),
    ("RQ2: score x female owner", "log_sales ~ score * female + score * credit + log_emp" + FE,
     ["log_sales", "score", "female", "credit", "log_emp"], "score:female", 0.1, "difference per +0.1 score"),
    ("RQ3: score x credit obstacle", "log_sales ~ score * female + score * credit + log_emp" + FE,
     ["log_sales", "score", "female", "credit", "log_emp"], "score:credit", 0.1, "difference per +0.1 score"),
    ("RQ4: score x female x credit", "log_sales ~ score * female * credit + log_emp" + FE,
     ["log_sales", "score", "female", "credit", "log_emp"], "score:female:credit", 0.1, "difference per +0.1 score")]
print("Tools ready.")

---
## Part 1. Main results for SMEs, under both size definitions
**The question:** do the main answers change when "SME" means *up to the national ceiling* (99–200 employees) or *under 100 employees* (World Bank)?

| Model | Equation (all with size and country-year effects) |
|---|---|
| RQ1 | $\ln(\text{Sales}) = \alpha + \beta\,\text{Score} + \gamma\ln(\text{Emp}) + \delta_{ct} + \varepsilon$ |
| Practices | $\ldots + \beta_1\text{Monitoring} + \beta_2\text{Targets} + \beta_3\text{Bonuses} + \ldots$ |
| RQ2, RQ3 | $\ldots + \beta_4(\text{Score}\times\text{Female}) + \beta_5(\text{Score}\times\text{Credit}) + \ldots$ |
| RQ4 | $\ldots + \beta_7(\text{Score}\times\text{Female}\times\text{Credit}) + \ldots$ |

**How to read the table:** each row is one result; the two column groups are the two definitions. **Evidence** uses the bootstrap p-value.

In [ ]:
rows, MAIN = [], {}
for label, f, need, term, step, kind in KEY:
    row = {"Result": label}
    for dname, data in SME.items():
        m, d = run(f, need, data)
        b = m.params[term]
        bp = boot_p(f, d, term)
        MAIN[(label, dname)] = {"b": b, "p": m.pvalues[term], "bp": bp, "n": int(m.nobs), "se": m.bse[term], "step": step}
        row.update({f"{dname}: firms": int(m.nobs), f"{dname}: result": f"{pct(b, step):+.1f}% {kind}",
                    f"{dname}: bootstrap p": round(bp, 4), f"{dname}: evidence": evidence(bp)})
    rows.append(row)
T1 = pd.DataFrame(rows)
add("1. Main results for SMEs: national ceilings vs World Bank 100+", T1,
    "Country-year fixed effects; robust standard errors; wild-cluster bootstrap p-values (Webb weights, clustered by survey).")

---
## Part 2. Weighted regressions
**Why:** survey weights make the results describe **all firms in the economy**, not only the firms the survey picked. Check 7 of the Question 1 notebook showed the weights are very uneven (7,418 firms carry the information of about 1,200), so the weighted results are **robustness checks** next to the unweighted main results.

| Version | What it does |
|---|---|
| **Unweighted** | Main result (as in Part 1) |
| **Rescaled within survey** | Weights corrected so no country dominates (mean weight 1 in each survey) |
| **Rescaled and capped** | The same, with the most extreme weights capped at the 99th percentile |

**How to read it:** if the three versions give similar answers, the survey's over-sampling of bigger firms does not change the conclusion. p-values here are robust (the bootstrap does not handle weights).

In [ ]:
for dname, data in SME.items():
    rows = []
    for label, f, need, term, step, kind in KEY:
        row = {"Result": label}
        for wname, wcol in WEIGHTS.items():
            m, d = run(f, need, data, wcol)
            b, p = m.params[term], m.pvalues[term]
            row[wname] = f"{pct(b, step):+.1f}% ({evidence(p)}, p {p:.3f})"
            if label == "RQ1: management score":
                SUMMARY[f"w_{dname}_{wname}"] = (pct(b, step), p)
        rows.append(row)
    add(f"2. Weighted vs unweighted, SMEs ({dname})", pd.DataFrame(rows),
        "Each cell: % reading (evidence, robust p-value).")

---
## Part 3. Indonesia + Malaysia vs the other surveys
**Why:** Indonesia and Malaysia supply about two-thirds of the firms with management answers. Is the payoff of management the same there as in the other 9 surveys?

$$\ln(\text{Sales}) = \alpha + \beta_1\,\text{Score} + \beta_2\,(\text{Score}\times\text{IdnMys}) + \gamma\ln(\text{Emp}) + \delta_{ct} + \varepsilon$$

$\beta_1$ is the payoff in the other surveys, $\beta_1+\beta_2$ in Indonesia + Malaysia, and **$\beta_2$ the difference** (bootstrap p over all 12 surveys). The country itself is absorbed by $\delta_{ct}$.

**Two samples:** all SMEs, and SMEs with **20+ employees**, the like-for-like comparison (Check 8 showed the groups differ mainly because Indonesia and Malaysia 2019 asked small firms too).

In [ ]:
f3 = "log_sales ~ score + score:indonesia_malaysia + log_emp" + FE
need3 = ["log_sales", "score", "log_emp", "indonesia_malaysia"]
rows = []
for sname, cond in [("All SMEs", lambda x: x), ("SMEs with 20+ employees", lambda x: x[x["l1"] >= 20])]:
    for dname, data in SME.items():
        m, d = run(f3, need3, cond(data))
        b_ot, p_ot = m.params["score"], m.pvalues["score"]
        b_im, p_im = combo(m, {"score": 1, "score:indonesia_malaysia": 1})
        bdiff = m.params["score:indonesia_malaysia"]
        bp = boot_p(f3, d, "score:indonesia_malaysia")
        rows.append({"Sample": sname, "Definition": dname,
                     "Firms: Indonesia + Malaysia": int(d["indonesia_malaysia"].sum()), "Firms: other surveys": int((1 - d["indonesia_malaysia"]).sum()),
                     "Indonesia + Malaysia: % per +0.1 score": f"{pct(b_im, 0.1):+.1f}% ({evidence(p_im)})",
                     "Other surveys: % per +0.1 score": f"{pct(b_ot, 0.1):+.1f}% ({evidence(p_ot)})",
                     "Difference": f"{pct(bdiff, 0.1):+.1f}%", "Difference bootstrap p": round(bp, 4), "Evidence of a difference": evidence(bp)})
        SUMMARY[f"im_{sname}_{dname}"] = (pct(b_im, 0.1), pct(b_ot, 0.1), bp, p_im, p_ot)
T3 = pd.DataFrame(rows)
add("3. Indonesia + Malaysia vs the other surveys (SMEs)", T3)

---
## Part 4. SMEs vs large firms
$$\ln(\text{Sales}) = \alpha + \beta_1\,\text{Score} + \beta_2\,\text{Large} + \beta_3\,(\text{Score}\times\text{Large}) + \gamma_1\ln(\text{Emp}) + \gamma_2\,(\ln\text{Emp}\times\text{Large}) + \delta_{ct} + \varepsilon$$

$\beta_1$ is the payoff for SMEs, $\beta_1+\beta_3$ for large firms, and **$\beta_3$ the difference** (bootstrap p). Uses all firms in the final sample. (The Question 1 notebook, Part R, adds more checks: three size groups and size as a continuous number.)

In [ ]:
rows = []
for dname, lcol in DEFS.items():
    f4 = f"log_sales ~ score * {lcol} + log_emp * {lcol}" + FE
    m, d = run(f4, ["log_sales", "score", "log_emp", lcol], D)
    b_s, p_s = m.params["score"], m.pvalues["score"]
    b_l, p_l = combo(m, {"score": 1, f"score:{lcol}": 1})
    bdiff = m.params[f"score:{lcol}"]
    bp = boot_p(f4, d, f"score:{lcol}")
    rows.append({"Definition": dname, "SMEs": int((d[lcol] == 0).sum()), "Large firms": int((d[lcol] == 1).sum()),
                 "SMEs: % per +0.1 score": f"{pct(b_s, 0.1):+.1f}% ({evidence(p_s)})",
                 "Large firms: % per +0.1 score": f"{pct(b_l, 0.1):+.1f}% ({evidence(p_l)})",
                 "Difference": f"{pct(bdiff, 0.1):+.1f}%", "Difference bootstrap p": round(bp, 4), "Evidence of a difference": evidence(bp)})
    SUMMARY[f"size_{dname}"] = (pct(b_s, 0.1), pct(b_l, 0.1), bp, p_s, p_l)
T4 = pd.DataFrame(rows)
add("4. SMEs vs large firms: payoff of management", T4)

---
## Part 5. The main payoff across every version
Each line is the payoff of a +0.1 management score (RQ1) in one version, with its 95% range. If the lines are close together and stay to the right of 0, the main result does not depend on these choices.

In [ ]:
pts = []
for dname, data in SME.items():
    for wname, wcol in WEIGHTS.items():
        m, d = run(KEY[0][1], KEY[0][2], data, wcol)
        pts.append((f"SMEs, {dname}, {wname.lower()}", m.params["score"], m.bse["score"]))
    for sname, sub in [("Indonesia + Malaysia", data[data["indonesia_malaysia"] == 1]), ("other surveys", data[data["indonesia_malaysia"] == 0])]:
        m, d = run(KEY[0][1], KEY[0][2], sub)
        pts.append((f"SMEs, {dname}, {sname} only", m.params["score"], m.bse["score"]))
for dname, lcol in DEFS.items():
    m, d = run(KEY[0][1], KEY[0][2], D[D[lcol] == 1])
    pts.append((f"Large firms, {dname}", m.params["score"], m.bse["score"]))
fig, ax = plt.subplots(figsize=(8.5, 0.42 * len(pts) + 1))
for k, (lab, b, se) in enumerate(pts):
    ax.errorbar(pct(b, 0.1), k, xerr=[[pct(b, 0.1) - pct(b - 1.96 * se, 0.1)], [pct(b + 1.96 * se, 0.1) - pct(b, 0.1)]],
                fmt="o", color="#1f3864" if lab.startswith("SMEs") else "#c62828", capsize=3)
ax.set_yticks(range(len(pts))); ax.set_yticklabels([p[0] for p in pts], fontsize=9); ax.invert_yaxis()
ax.axvline(0, color="grey", linewidth=1)
ax.set_xlabel("% more sales per +0.1 management score (95% range)")
fig.tight_layout()
add_figure("5. Payoff of management (RQ1) across every version", fig,
           "Blue: SMEs; red: large firms (separate regressions). 95% ranges use robust standard errors.")

---
## Part 6. Formal training (supplementary analysis)
**Two questions:**
1. Do firms that run **formal training** for their workers sell more than similar firms that don't?
2. Does management **pay off more** in firms that train? Training may be a **complement**: trained workers may make better use of targets, monitoring and bonuses.

$$\ln(\text{Sales}) = \alpha + \beta_1\,\text{Score} + \beta_2\,\text{Training} + \beta_3\,(\text{Score}\times\text{Training}) + \gamma\ln(\text{Emp}) + \delta_{ct} + \varepsilon$$

$\beta_1$ is the payoff of management in firms **without** training, $\beta_1+\beta_3$ **with** training, and **$\beta_3$ the difference** (bootstrap p). Training = 1 if the firm ran formal training programmes for its permanent full-time workers (question l10). This is a supplementary analysis, not a numbered hypothesis.

In [ ]:
rows = []
need_t = ["log_sales", "score", "training", "log_emp"]
for dname, data in SME.items():
    # (1) the link between training and sales, holding the management score
    fA = "log_sales ~ score + training + log_emp" + FE
    mA, dA = run(fA, need_t, data)
    bA, bpA = mA.params["training"], boot_p(fA, dA, "training")
    # (2) does the payoff of management differ with training?
    fB = "log_sales ~ score * training + log_emp" + FE
    mB, dB = run(fB, need_t, data)
    b_no, p_no = mB.params["score"], mB.pvalues["score"]
    b_yes, p_yes = combo(mB, {"score": 1, "score:training": 1})
    b_d, bp_d = mB.params["score:training"], boot_p(fB, dB, "score:training")
    n_tr = int(dB["training"].sum())
    rows += [
        {"Result": "Firms with formal training vs similar firms without", "Definition": dname, "Firms": int(mA.nobs),
         "Plain reading": f"{pct(bA, 1):+.1f}% sales", "p-value": round(bpA, 4), "Which p": "bootstrap", "Evidence": evidence(bpA)},
        {"Result": "Payoff of +0.1 score: firms WITHOUT training", "Definition": dname, "Firms": int(mB.nobs) - n_tr,
         "Plain reading": f"{pct(b_no, 0.1):+.1f}% sales", "p-value": round(p_no, 4), "Which p": "robust", "Evidence": evidence(p_no)},
        {"Result": "Payoff of +0.1 score: firms WITH training", "Definition": dname, "Firms": n_tr,
         "Plain reading": f"{pct(b_yes, 0.1):+.1f}% sales", "p-value": round(p_yes, 4), "Which p": "robust", "Evidence": evidence(p_yes)},
        {"Result": "Difference in payoff (with minus without)", "Definition": dname, "Firms": int(mB.nobs),
         "Plain reading": f"{pct(b_d, 0.1):+.1f}% difference per +0.1 score", "p-value": round(bp_d, 4), "Which p": "bootstrap", "Evidence": evidence(bp_d)}]
    SUMMARY[f"tr_{dname}"] = dict(link=pct(bA, 1), p_link=bpA, no=pct(b_no, 0.1), p_no=p_no, yes=pct(b_yes, 0.1), p_yes=p_yes,
                                  diff=pct(b_d, 0.1), p_diff=bp_d, n=int(mB.nobs), n_tr=n_tr)
T6 = pd.DataFrame(rows)
add("6. Formal training: link with sales, and with the payoff of management (SMEs)", T6,
    "Supplementary analysis. Country-year fixed effects; robust standard errors; bootstrap p where shown.")

---
## Part 7. Regression tables in journal format (stargazer)
The main regressions again, laid out as in a journal article: one column per model, the standard error in brackets under each coefficient, stars for significance, and lines for fixed effects, controls and the number of firms.

| Table | Content |
|---|---|
| **1** | Main results, national SME ceilings: score, score with controls, practices, gender and credit, three-way term |
| **2** | The same, World Bank definition (under 100 employees) |
| **3, 4** | Weighted regressions: no weights, rescaled, rescaled and capped (national; World Bank) |
| **5** | Indonesia + Malaysia against the other surveys |
| **6** | SMEs against large firms |
| **7** | Formal training |

**Reading the stars:** they use the **robust** p-values (\*p<0.1, \*\*p<0.05, \*\*\*p<0.01). With only 12 surveys the wild-cluster **bootstrap** p-values are more careful, so they are given in the note under each table.

The tables are shown below and saved as `q7_regression_tables.html` (all tables on one page) and `q7_regression_tables.tex` (for LaTeX).

In [ ]:
from stargazer.stargazer import Stargazer
try:
    from IPython.display import HTML, display
except Exception:
    HTML = lambda s: s

NAMES = {"score": "Management score", "monitoring": "Monitoring (r2)", "targets": "Targets (r4)", "bonuses": "Bonuses for managers (r8)",
         "female": "Female owner", "credit": "Credit problem", "score:female": "Score x Female owner", "score:credit": "Score x Credit problem",
         "female:credit": "Female x Credit problem", "score:female:credit": "Score x Female x Credit problem",
         "large": "Large firm", "score:large": "Score x Large firm", "log_emp:large": "Log employees x Large firm",
         "training": "Formal training", "score:training": "Score x Training", "score:indonesia_malaysia": "Score x Indonesia + Malaysia",
         "foreign": "Foreign-owned", "exporter": "Exporter", "log_age": "Log firm age", "log_emp": "Log employees"}
ORDER = ["score", "monitoring", "targets", "bonuses", "female", "credit", "score:female", "score:credit", "female:credit", "score:female:credit",
         "large", "score:large", "training", "score:training", "score:indonesia_malaysia", "foreign", "exporter", "log_age", "log_emp", "log_emp:large"]
BASE_NOTES = ["Robust (HC1) standard errors in brackets; stars use robust p-values.",
              "The outcome is the log of sales in US dollars. Firms are compared only within the same country and year."]
TABLES = []                                         # (title, html, latex)

def fit(formula, needed, data, wcol=None):
    return run(formula, needed, data, wcol)[0]      # run() is defined in Step 2

def boot_note(items):
    """Bootstrap p-values as short lines (three per line), so the table does not become too wide."""
    parts = [f"{label} {p:.3f}" for label, p in items]
    lines = ["; ".join(parts[i:i + 3]) for i in range(0, len(parts), 3)]
    lines[0] = "Wild-cluster bootstrap p-values (12 surveys): " + lines[0]
    return [l + ("." if k == len(lines) - 1 else ";") for k, l in enumerate(lines)]

def stargazer_table(title, models, columns, separators, lines, notes):
    sg = Stargazer(models)
    sg.title(title)
    sg.dependent_variable_name("Log sales (US dollars)")      # no $ sign: it would start math mode in LaTeX
    sg.custom_columns(columns, separators)
    sg.covariate_order([c for c in ORDER if any(c in m.params.index for m in models)])
    sg.rename_covariates(NAMES)
    sg.add_line("Country-year fixed effects", ["Yes"] * len(models))
    for label, values in lines:
        sg.add_line(label, values)
    sg.add_custom_notes(BASE_NOTES + notes)
    sg.show_adj_r2 = False; sg.show_residual_std_err = False; sg.show_f_statistic = False
    sg.significant_digits(3)
    TABLES.append((title, sg.render_html(), sg.render_latex()))

# ---- Tables 1 and 2: main results, one table per size definition
for n, (dname, data) in enumerate(SME.items(), 1):
    models = [fit(KEY[i][1], KEY[i][2], data) for i in (0, 1, 2, 5, 7)]          # score, + controls, practices, RQ2 and RQ3, RQ4
    bp = lambda i: MAIN[(KEY[i][0], dname)]["bp"]
    stargazer_table(f"Management and sales, SMEs ({dname})", models,
                    ["Score", "Score + controls", "Practices", "Gender and credit", "Three-way"], [1, 1, 1, 1, 1],
                    [("Controls", ["No", "Yes", "No", "No", "No"])],
                    boot_note([("score", bp(0)), ("score with controls", bp(1)), ("monitoring", bp(2)), ("targets", bp(3)), ("bonuses", bp(4)),
                                ("score x female", bp(5)), ("score x credit", bp(6)), ("score x female x credit", bp(7))]))

# ---- Tables 3 and 4: weighted regressions
for n, (dname, data) in enumerate(SME.items(), 3):
    models = [fit(KEY[i][1], KEY[i][2], data, wcol) for i in (0, 2) for wcol in WEIGHTS.values()]
    stargazer_table(f"Weighted regressions, SMEs ({dname})", models,
                    ["Management score", "Practices"], [3, 3],
                    [("Weights", ["None", "Rescaled", "Capped"] * 2)],
                    ["Weighted least squares; p-values are robust, not bootstrapped.", "Rescaled: weights with an average of 1 in each survey. Capped: rescaled, then capped at the 99th percentile."])

# ---- Table 5: Indonesia + Malaysia against the other surveys
f_im = "log_sales ~ score + score:indonesia_malaysia + log_emp" + FE
need_im = ["log_sales", "score", "log_emp", "indonesia_malaysia"]
models, samples, items = [], [], []
for dname, data in SME.items():
    for sname, sub in [("All SMEs", data), ("SMEs with 20+ employees", data[data["l1"] >= 20])]:
        models.append(fit(f_im, need_im, sub))
        samples.append("20+ employees" if "20+" in sname else "All SMEs")
        items.append((f"{samples[-1]}, {dname}", SUMMARY[f"im_{sname}_{dname}"][2]))
stargazer_table("Indonesia + Malaysia against the other surveys (SMEs)", models,
                ["National ceilings", "World Bank 100+"], [2, 2], [("Sample", samples)],
                ["Score is the payoff in the other surveys; Score x Indonesia + Malaysia is the difference."] + boot_note(items))

# ---- Table 6: SMEs against large firms
models, items = [], []
for dname, lcol in DEFS.items():
    d_ = D.assign(large=D[lcol])
    models.append(fit("log_sales ~ score * large + log_emp * large" + FE, ["log_sales", "score", "log_emp", "large"], d_))
    items.append((dname, SUMMARY[f"size_{dname}"][2]))
stargazer_table("SMEs against large firms", models, list(DEFS), [1, 1], [],
                ["Score is the payoff for SMEs; Score x Large firm is the difference for large firms."] + boot_note(items))

# ---- Table 7: formal training
models, items = [], []
for dname, data in SME.items():
    need_t = ["log_sales", "score", "training", "log_emp"]
    models += [fit("log_sales ~ score + training + log_emp" + FE, need_t, data), fit("log_sales ~ score * training + log_emp" + FE, need_t, data)]
    items += [(f"training, {dname}", SUMMARY[f"tr_{dname}"]["p_link"]), (f"score x training, {dname}", SUMMARY[f"tr_{dname}"]["p_diff"])]
stargazer_table("Formal training (SMEs)", models, ["National ceilings", "World Bank 100+"], [2, 2], [],
                ["Training is a yes or no variable; Score x Training is the difference in the payoff of management."] + boot_note(items))

# ---- Save and show
CSS = ("body{font-family:Georgia,'Times New Roman',serif;max-width:1100px;margin:28px auto;padding:0 18px;color:#1d1d1d}"
       "h1{font-size:26px}.tbl{margin:34px 0;overflow-x:auto}.tbl table{margin:8px auto;font-size:14px;min-width:60%}"
       ".tbl td{padding:2px 12px}p.lead{color:#444}p.tn{font-weight:bold;margin:0}")
page = ("<!DOCTYPE html><html><head><meta charset='utf-8'><title>Regression tables</title><style>" + CSS + "</style></head><body>"
        "<h1>Regression tables</h1><p class='lead'>Journal-style tables made with stargazer. Standard errors are in brackets. Stars use robust p-values; "
        "the wild-cluster bootstrap p-values are in the note under each table.</p>" + "".join(f'<div class="tbl"><p class="tn">Table {i}</p>{h}</div>' for i, (_, h, _) in enumerate(TABLES, 1)) + "</body></html>")
(OUT / "q7_regression_tables.html").write_text(page, encoding="utf-8")
(OUT / "q7_regression_tables.tex").write_text("\n\n".join(l for _, _, l in TABLES), encoding="utf-8")
for i, (_, h, _) in enumerate(TABLES, 1):
    display(HTML(f"<p><b>Table {i}</b></p>" + h))
print("Saved", OUT / "q7_regression_tables.html", "and", OUT / "q7_regression_tables.tex")

---
## Summary
Every number is calculated above; each reading is chosen from the bootstrap p-value.

In [ ]:
def reading(p):
    return "Clear (p < 0.05)" if p < 0.05 else "Weak (p < 0.10)" if p < 0.10 else "No clear evidence"
rows = []
for label in [k[0] for k in KEY]:
    for dname in DEFS:
        r = MAIN[(label, dname)]
        rows.append({"Question": label if dname == "National ceilings" else "", "Definition": dname, "Firms": r["n"],
                     "Result": f"{pct(r['b'], r['step']):+.1f}%", "Bootstrap p": round(r["bp"], 4), "Reading": reading(r["bp"])})
for dname in DEFS:
    u, rr, c = (SUMMARY[f"w_{dname}_{w}"] for w in WEIGHTS)
    rows.append({"Question": "Weights (RQ1)" if dname == "National ceilings" else "", "Definition": dname, "Firms": "",
                 "Result": f"unweighted {u[0]:+.1f}%, rescaled {rr[0]:+.1f}%, capped {c[0]:+.1f}%", "Bootstrap p": "",
                 "Reading": "Similar" if max(u[0], rr[0], c[0]) - min(u[0], rr[0], c[0]) < 3 else "Differs: see Part 2"})
for dname in DEFS:
    im, ot, bp = SUMMARY[f"im_SMEs with 20+ employees_{dname}"][:3]
    rows.append({"Question": "Indonesia + Malaysia vs rest (20+)" if dname == "National ceilings" else "", "Definition": dname, "Firms": "",
                 "Result": f"{im:+.1f}% vs {ot:+.1f}%", "Bootstrap p": round(bp, 4), "Reading": reading(bp)})
for dname in DEFS:
    s, l, bp = SUMMARY[f"size_{dname}"][:3]
    rows.append({"Question": "SMEs vs large firms" if dname == "National ceilings" else "", "Definition": dname, "Firms": "",
                 "Result": f"SMEs {s:+.1f}% vs large {l:+.1f}%", "Bootstrap p": round(bp, 4), "Reading": reading(bp)})
for dname in DEFS:
    t = SUMMARY[f"tr_{dname}"]
    rows.append({"Question": "Training (supplementary)" if dname == "National ceilings" else "", "Definition": dname, "Firms": t["n"],
                 "Result": f"training {t['link']:+.1f}%; payoff without {t['no']:+.1f}% vs with {t['yes']:+.1f}%",
                 "Bootstrap p": round(t["p_diff"], 4), "Reading": reading(t["p_diff"])})
add("Summary. All supervisor regressions", pd.DataFrame(rows),
    "Percentages: per +0.1 management score, or vs similar firms without the practice. 'Reading' for comparisons refers to the difference.")

### Save the results
Writes every table and the chart (summary first) to an HTML report and an Excel file in `wbes_out/sea_data/q7/`.

In [ ]:
import html as _html
order = ["Summary. All supervisor regressions"] + [t for t in REPORT if not t.startswith("Summary")]
CSS = ("body{font-family:Georgia,'Times New Roman',serif;max-width:1150px;margin:28px auto;padding:0 18px;color:#1d1d1d}"
       "h1{font-size:26px}h2{font-size:19px;border-bottom:1px solid #ccc;padding-bottom:4px;margin-top:34px}p{color:#444}"
       "table.t{border-collapse:collapse;font-size:12.5px;margin:6px 0 4px}table.t th{background:#1f3864;color:#fff;padding:5px 8px;text-align:left}"
       "table.t td{border-bottom:1px solid #ddd;padding:4px 8px;vertical-align:top}table.t tr:nth-child(even) td{background:#f4f6fa}.wrap{overflow-x:auto}")
is_fig = lambda item: isinstance(item[0], str) and item[0] == "__figure__"
parts = []
for i, title in enumerate(order):
    item = REPORT[title]
    body = (f'<img style="max-width:100%" src="data:image/png;base64,{item[2]}">' if is_fig(item)
            else '<div class="wrap">' + item[0].to_html(index=False, border=0, classes="t", na_rep="") + "</div>")
    parts.append(f'<section id="s{i}"><h2>{_html.escape(title)}</h2>' + (f"<p>{_html.escape(item[1])}</p>" if item[1] else "") + body + "</section>")
toc = "<ol>" + "".join(f'<li><a href="#s{i}">{_html.escape(t)}</a></li>' for i, t in enumerate(order)) + "</ol>"
page = (f"<!DOCTYPE html><html><head><meta charset='utf-8'><title>Supervisor regressions</title><style>{CSS}</style></head>"
        f"<body><h1>Regressions following the supervisor's suggestions</h1>"
        f"<p>{len(D):,} firms in the final sample (management score and believable sales), from sea_clean_firms.csv. "
        f"Firms are compared within the same country and year; results show differences between similar firms, not cause and effect.</p>"
        f"<h2>Contents</h2>{toc}{''.join(parts)}</body></html>")
(OUT / "q7_supervisor_regressions.html").write_text(page, encoding="utf-8")
with pd.ExcelWriter(OUT / "q7_supervisor_regressions.xlsx") as xw:
    j = 0
    for title in order:
        if is_fig(REPORT[title]):
            continue
        j += 1
        REPORT[title][0].to_excel(xw, sheet_name=f"{j:02d} " + re.sub(r"[\[\]:*?/\\']", "", title)[:27], index=False)
print("Saved to", OUT)
try:
    from google.colab import files
    files.download(str(OUT / "q7_supervisor_regressions.html"))
except Exception:
    pass

### A simple results page
Writes `q7_results_simple.html`: the main results in plain words, with simple bars and colour tags (Clear / Weak / No clear evidence). Every number comes from the steps above.

In [ ]:
import html as _h
NAT, WB = "National ceilings", "World Bank 100+"

def tag(p):
    """Colour tag from a p-value: clear (< 0.05), weak (< 0.10), or no clear evidence."""
    if p is None or pd.isna(p):
        return "plain", ""
    return ("clear", "Clear") if p < 0.05 else ("weak", "Weak") if p < 0.10 else ("none", "Not clear")

def row(label, value, vmax, p, sub=""):
    cls, word = tag(p)
    w = max(1.5, min(100, 100 * abs(value) / vmax))
    return (f'<div class="row"><div class="lab">{_h.escape(label)}' + (f'<small>{_h.escape(sub)}</small>' if sub else "") + '</div>'
            f'<div class="track"><span class="fill {cls}{" neg" if value < 0 else ""}" style="width:{w:.1f}%"></span></div>'
            f'<div class="val">{value:+.1f}%</div><div class="tag {cls}">{word}</div></div>')

def section(title, answer, rows_html, note=""):
    return (f'<section><h2>{_h.escape(title)}</h2><p class="answer">{answer}</p>{rows_html}'
            + (f'<p class="note">{note}</p>' if note else "") + '</section>')

vmax = lambda vals: max(abs(v) for v in vals) * 1.08

# ---- 1. The headline: management and sales
r1 = MAIN[("RQ1: management score", NAT)]
ladder = [(s, pct(r1["b"], s)) for s in (0.2, 0.4, 0.6, 0.8)]
lad = "".join(f'<div class="rung"><div class="rbar" style="height:{max(4, 100 * v / ladder[-1][1]):.0f}%"><span>{v:+.0f}%</span></div>'
              f'<div class="rlab">+{s:.1f}</div></div>' for s, v in ladder)

# ---- 2. Practices
prac = [("Bonuses for managers", "Bonuses for managers (r8)"), ("Targets", "Targets (r4)"), ("Monitoring", "Monitoring (r2)")]
pv = [MAIN[(k, NAT)] for _, k in prac]
s2 = "".join(row(lbl, pct(r["b"], 1), vmax([pct(x["b"], 1) for x in pv]), r["bp"], "vs similar firms without it")
             for (lbl, _), r in zip(prac, pv))
PHRASE = {"Bonuses for managers": "performance bonuses for managers", "Targets": "production targets",
          "Monitoring": "monitoring of performance indicators"}
cands = [(lbl, r) for (lbl, _), r in zip(prac, pv) if r["b"] > 0]
best = min(cands, key=lambda x: x[1]["bp"]) if cands else None
if best and best[1]["bp"] < 0.05:
    prac_ans = f"Of the three practices, <b>{PHRASE[best[0]]}</b> show the clearest link with sales."
    prac_mean = f"<b>{best[0]} stand out:</b> firms with {PHRASE[best[0]]} sell {pct(best[1]['b'], 1):+.0f}% more than similar firms without."
elif best and best[1]["bp"] < 0.10:
    prac_ans = f"No practice is clearly linked with sales on its own; <b>{PHRASE[best[0]]}</b> come closest (weak evidence)."
    prac_mean = f"<b>No single practice stands out clearly</b>; {PHRASE[best[0]]} come closest."
else:
    prac_ans = "None of the three practices shows a clear link with sales on its own."
    prac_mean = "<b>No single practice stands out clearly</b> on its own."

# ---- 3. Training
t = SUMMARY[f"tr_{NAT}"]
tv = [t["link"], t["no"], t["yes"]]
s3 = (row("Firms with formal training", t["link"], vmax(tv), t["p_link"], "vs similar firms without training")
      + row("Payoff of +0.1 score, no training", t["no"], vmax(tv), t["p_no"], f"{t['n'] - t['n_tr']:,} firms")
      + row("Payoff of +0.1 score, with training", t["yes"], vmax(tv), t["p_yes"], f"{t['n_tr']:,} firms"))
tr_ans = (f"Firms that train their workers sell {t['link']:+.1f}% compared with similar firms. "
          + ("Management pays off <b>more</b> in firms that train." if (t["p_diff"] < 0.05 and t["diff"] > 0)
             else "Management pays off <b>less</b> in firms that train." if (t["p_diff"] < 0.05 and t["diff"] < 0)
             else "Management pays off about <b>the same</b> with or without training"
             + f" (difference {t['diff']:+.1f}%, not significant)."))

# ---- 4. Gender and credit
gc = [("Female owner", "RQ2: score x female owner"), ("Credit obstacle", "RQ3: score x credit obstacle"),
      ("Female owner with a credit obstacle", "RQ4: score x female x credit")]
gv = [MAIN[(k, NAT)] for _, k in gc]
s4 = "".join(row(lbl, pct(r["b"], 0.1), max(5, vmax([pct(x["b"], 0.1) for x in gv])), r["bp"], "difference in the payoff of +0.1 score")
             for (lbl, _), r in zip(gc, gv))
any_gc = any(r["bp"] < 0.05 for r in gv)

# ---- 5. Does the answer hold everywhere?
robust = []
for dname, short in [(NAT, "national SME ceilings"), (WB, "World Bank 100+")]:
    for wname, wshort in [("Unweighted", "no weights"), ("Weights rescaled within survey", "rescaled weights"),
                          ("Rescaled and capped weights", "capped weights")]:
        v, p = SUMMARY[f"w_{dname}_{wname}"]
        robust.append((f"SMEs, {short}", wshort, v, p))
im, ot, _, p_im, p_ot = SUMMARY[f"im_SMEs with 20+ employees_{NAT}"]
robust += [("Indonesia + Malaysia", "SMEs with 20+ employees", im, p_im), ("Other 9 surveys", "SMEs with 20+ employees", ot, p_ot)]
for dname, short in [(NAT, "national SME ceilings"), (WB, "World Bank 100+")]:
    sv, lv, _, ps, pl = SUMMARY[f"size_{dname}"]
    robust += [("SMEs", short, sv, ps), ("Large firms", short, lv, pl)]
s5 = "".join(row(a, v, vmax([x[2] for x in robust]), p, b) for a, b, v, p in robust)
n_pos = sum(1 for x in robust if x[2] > 0 and x[3] < 0.05)
hold = "Yes." if n_pos == len(robust) else "Mostly." if n_pos >= 0.75 * len(robust) else "Only partly."

S1 = f"""<header><p class="kicker">Results for {len(D):,} firms in {D["survey"].nunique()} surveys from {D["country"].nunique()} Southeast Asian countries</p>
<h1>{"Better-managed SMEs sell more" if (r1["b"] > 0 and r1["bp"] < 0.10) else "Management practices and SME sales"}</h1>
<p class="lead">Among SMEs of the same size, in the same country and year, a management score <b>0.1 higher</b>
goes with about <b>{pct(r1['b'], 0.1):+.1f}% more sales</b>. The evidence is <span class="tag {tag(r1['bp'])[0]}">{tag(r1['bp'])[1]}</span></p>
<p class="ladcap">Extra sales when the management score is higher by:</p>
<div class="ladder" aria-label="Extra sales as the management score rises">{lad}</div>
<p class="note">How to read the bars: extra sales compared with a similar SME whose score is lower by the amount shown (score from 0 = no practices to about 1 = all practices).</p></header>"""

legend = """<section class="legend"><h2>How sure are we?</h2>
<p>Each result has a colour tag, based on how likely it is to be chance:</p>
<p><span class="tag clear">Clear</span> very unlikely to be chance (p &lt; 0.05)</p>
<p><span class="tag weak">Weak</span> some evidence (p &lt; 0.10)</p>
<p><span class="tag none">Not clear</span> could easily be chance</p>
<p class="note">p-values use the wild-cluster bootstrap over the 12 surveys where possible; weighted results use robust p-values.</p></section>"""

means = f"""<section><h2>What it means</h2><ul>
<li><b>Management matters:</b> better-managed SMEs sell more than similar SMEs, and this holds in {n_pos} of {len(robust)} versions tested.</li>
<li>{prac_mean}</li>
<li><b>Gender and credit {'change' if any_gc else 'do not change'} the payoff:</b> {'see the section above' if any_gc else 'management helps female- and male-owned firms, with or without credit problems, about equally'}.</li>
<li><b>A caution:</b> these are differences between similar firms. They do not prove that adopting a practice would raise a firm's sales.</li></ul></section>"""

CSS = """
:root{--ink:#1f2937;--soft:#5b6472;--line:#e3e7ec;--bg:#ffffff;--track:#eef1f4;--clear:#0f766e;--weak:#b45309;--none:#8a94a3;--panel:#f7f9fb}
@media (prefers-color-scheme:dark){:root{--ink:#e6e9ee;--soft:#a3acb9;--line:#2c333d;--bg:#14181e;--track:#232a33;--clear:#2bb3a3;--weak:#e0a050;--none:#7d8796;--panel:#1a1f27}}
*{box-sizing:border-box}body{margin:0;background:var(--bg);color:var(--ink);font:17px/1.6 "Atkinson Hyperlegible",system-ui,-apple-system,"Segoe UI",Roboto,sans-serif}
main{max-width:760px;margin:0 auto;padding:40px 22px 64px}
h1,h2{font-weight:700;line-height:1.2;letter-spacing:-0.01em}
h1{font-size:40px;margin:6px 0 14px}h2{font-size:24px;margin:0 0 6px}
.kicker{color:var(--soft);font-size:14px;margin:0}.lead{font-size:20px;margin:0 0 22px}
section{border-top:1px solid var(--line);padding:28px 0 6px}.answer{font-size:18px;margin:0 0 16px}
.note{color:var(--soft);font-size:14px}
.ladder{display:flex;gap:18px;align-items:flex-end;height:190px;padding:10px 0 0;border-bottom:2px solid var(--line)}
.rung{flex:1;height:100%;display:flex;flex-direction:column;justify-content:flex-end;align-items:stretch}
.rbar{background:var(--clear);border-radius:6px 6px 0 0;position:relative;min-height:6px}
.rbar span{position:absolute;top:-26px;left:0;right:0;text-align:center;font-weight:700;color:var(--ink)}
.ladcap{font-weight:700;margin:0 0 34px}
.rlab{text-align:center;color:var(--soft);font-size:14px;padding-top:6px}
.row{display:grid;grid-template-columns:minmax(190px,1.7fr) 2fr 76px 92px;gap:12px;align-items:center;padding:8px 0;border-bottom:1px solid var(--line)}
.lab{font-weight:600}.lab small{display:block;font-weight:400;color:var(--soft);font-size:13px}
.track{background:var(--track);height:14px;border-radius:7px;overflow:hidden}
.fill{display:block;height:100%;border-radius:7px}.fill.clear{background:var(--clear)}.fill.weak{background:var(--weak)}.fill.none,.fill.plain{background:var(--none)}
.fill.neg{background-image:repeating-linear-gradient(45deg,transparent 0 4px,rgba(255,255,255,.45) 4px 8px)}
.val{font-weight:700;text-align:right;font-variant-numeric:tabular-nums}
.tag{display:inline-block;white-space:nowrap;font-size:13px;font-weight:600;padding:2px 10px;border-radius:999px;color:#fff;text-align:center}
.tag.clear{background:var(--clear)}.tag.weak{background:var(--weak)}.tag.none{background:var(--none)}.tag.plain{background:transparent}
.legend p{margin:6px 0}ul{padding-left:20px}li{margin:8px 0}
footer{border-top:1px solid var(--line);margin-top:26px;padding-top:16px;color:var(--soft);font-size:13px}
@media (max-width:620px){h1{font-size:31px}.row{grid-template-columns:1fr 64px;}.track{grid-column:1/-1;grid-row:2}.tag{grid-column:1/-1;justify-self:start}}
"""

page = f"""<!DOCTYPE html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">
<title>Results in plain words</title>
<link rel="preconnect" href="https://fonts.googleapis.com"><link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:wght@400;700&display=swap" rel="stylesheet"><style>{CSS}</style></head><body><main>
{S1}
{legend}
{section("Which practices matter most?", prac_ans, s2)}
{section("Does training help?", tr_ans, s3, "Supplementary analysis. Training = formal training programmes for permanent full-time workers.")}
{section("Do owner gender or credit problems change the payoff?",
         ("No. " if not any_gc else "Partly. ") + "The bars show how much the payoff of a 0.1 higher score differs for each group; a bar near zero means no difference.", s4)}
{section("Does the answer hold everywhere?",
         f"{hold} The payoff of a 0.1 higher score is positive and clear in <b>{n_pos} of {len(robust)}</b> versions: both size definitions, with and without survey weights, in Indonesia + Malaysia and the other countries, and in SMEs and large firms.", s5,
         "Weighted results and group payoffs use robust p-values; SMEs vs large firms come from one model with both groups.")}
{means}
<footer>Data: World Bank Enterprise Surveys, {D["survey"].nunique()} surveys with the management module ({int(D["year"].min())} to {int(D["year"].max())}). Management score: 8 questions scored 0 to 1 and averaged (a skipped follow-up after a No counts as 0).
Method: regression of log sales on the score, holding firm size and country-year fixed; robust standard errors and wild-cluster bootstrap p-values. Full tables: q7_supervisor_regressions.html.</footer>
</main></body></html>"""
(OUT / "q7_results_simple.html").write_text(page, encoding="utf-8")
print("Saved", OUT / "q7_results_simple.html")
try:
    from google.colab import files
    files.download(str(OUT / "q7_results_simple.html"))
except Exception:
    pass